# Pooled FastText features: LR vs LightGBM (RQ2)

FastText `wiki-news-300d-1M` (frozen) pooled into one vector per comment, then logistic regression or LightGBM, trained and evaluated on the shared 5 stratified folds with `comment_heavy` as input.

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = Path("/kaggle/working").exists()

if IN_COLAB or IN_KAGGLE:
    if IN_KAGGLE:
        %cd /kaggle/working
    !git clone -q https://github.com/mnhhuyen/AI4SE_toxic_comments.git
    %cd AI4SE_toxic_comments
    !pip install -q -e ".[dev]" lightgbm tabulate

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

# Kaggle: attach the Jigsaw competition data, optionally a dataset with wiki-news-300d-1M.vec
FASTTEXT = ROOT / "data" / "embeddings" / "fasttext" / "wiki-news-300d-1M.vec"
RAW_TRAIN = ROOT / "data" / "raw" / "train.csv"
if IN_KAGGLE:
    import pandas as pd

    attached = next(Path("/kaggle/input").rglob("wiki-news-300d-1M.vec"), None)
    if attached and not FASTTEXT.exists():
        FASTTEXT.symlink_to(attached)
    if not RAW_TRAIN.exists():
        RAW_TRAIN.parent.mkdir(parents=True, exist_ok=True)
        pd.read_csv(next(Path("/kaggle/input").rglob("train.csv*"))).to_csv(RAW_TRAIN, index=False)

if (IN_COLAB or IN_KAGGLE) and not FASTTEXT.exists():
    !wget -q -c https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip -P {FASTTEXT.parent}
    !unzip -q -n {FASTTEXT.parent}/wiki-news-300d-1M.vec.zip -d {FASTTEXT.parent}

## Data and folds

In [ ]:
import pandas as pd

from toxic_comments.cleaning import process_cleaning, validate_training_data
from toxic_comments.config import (ID_COLUMN, K_FOLD_DATA_DIR, LABEL_COLUMNS, MODELS_DIR,
                                   RAW_DATA_DIR, RESULTS_DIR)
from toxic_comments.splits import save_kfold_datasets

TEXT_COLUMNS = ["comment_text", "comment_light", "comment_heavy"]

if not list(K_FOLD_DATA_DIR.glob("fold_*/test.csv")):
    data = process_cleaning(validate_training_data(pd.read_csv(RAW_DATA_DIR / "train.csv")))
    data[TEXT_COLUMNS] = data[TEXT_COLUMNS].fillna("").astype(str)
    data = data[data["is_empty_heavy"] == 0].reset_index(drop=True)
    save_kfold_datasets(data[[ID_COLUMN, *TEXT_COLUMNS, *LABEL_COLUMNS]], n_splits=5, strategy="stratified")

sorted(path.name for path in K_FOLD_DATA_DIR.glob("fold_*"))

## Training

`POOLING` is the configuration selected in RQ1 (FastText + max + supervised attention). Results go to `results/pooled/<pooling>/` so runs with different poolings do not overwrite each other.

Per fold, the embeddings step is fitted once and its features are shared by LR and LightGBM: tokenizing (CPU), then attention training and pooling (GPU), then the classifiers (CPU).

In [ ]:
import torch

from toxic_comments.train import train_pooled_from_folds

POOLING = "max_supervised_attention"
MODELS = ["fasttext_logistic_regression", "fasttext_lightgbm"]
RUN_RESULTS_DIR = RESULTS_DIR / "pooled" / POOLING
RUN_MODELS_DIR = MODELS_DIR / "pooled" / POOLING

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
pending = [name for name in MODELS if not (RUN_RESULTS_DIR / f"{name}_fold_metrics.csv").exists()]
print("skipping (results found):", [name for name in MODELS if name not in pending])
if pending:
    train_pooled_from_folds(folds_dir=K_FOLD_DATA_DIR, model_names=pending, output_dir=RUN_MODELS_DIR,
                            results_dir=RUN_RESULTS_DIR, embedding_pooling=POOLING)

## Results (mean ± std over 5 folds)

In [ ]:
METRICS = ["macro_roc_auc", "micro_roc_auc", "micro_f1", "macro_f1",
           "micro_precision", "micro_recall", "subset_accuracy", "hamming_loss"]

folds = pd.concat([pd.read_csv(RUN_RESULTS_DIR / f"{name}_fold_metrics.csv") for name in MODELS])
stats = folds.groupby("model_name")[METRICS].agg(["mean", "std"]).loc[MODELS]
summary = pd.DataFrame({
    metric: stats[(metric, "mean")].map("{:.4f}".format) + " ± " + stats[(metric, "std")].map("{:.4f}".format)
    for metric in METRICS
}).T
print(summary.to_markdown())

## Trainable parameters

In [ ]:
import joblib

fitted = {name: joblib.load(RUN_MODELS_DIR / "fold_1" / f"{name}.joblib") for name in MODELS}

vectorizer = fitted["fasttext_logistic_regression"].named_steps["embeddings"]
attention_params = sum(getattr(vectorizer, attr).size
                       for attr in ["attention_W_", "attention_b_", "attention_v_"]
                       if hasattr(vectorizer, attr))
lr = fitted["fasttext_logistic_regression"].named_steps["classifier"]
lr_params = sum(est.coef_.size + est.intercept_.size for est in lr.estimators_)
trees = sum(est.n_estimators_ for est in fitted["fasttext_lightgbm"].named_steps["classifier"].estimators_)

print(f"pooled dim: {vectorizer.output_dim_}")
print(f"attention params: {attention_params:,}")
print(f"LR: {lr_params:,} + attention = {lr_params + attention_params:,}")
print(f"LightGBM: {trees} trees + {attention_params:,} attention params")

## LaTeX rows for `tab:rq2-overall`

In [ ]:
TABLE_METRICS = ["macro_roc_auc", "micro_roc_auc", "micro_f1", "macro_f1"]
LABELS = {"fasttext_logistic_regression": "LR", "fasttext_lightgbm": "LightGBM"}

for name in MODELS:
    cells = " & ".join(f"{stats.loc[name, (m, 'mean')]:.4f} $\\pm$ {stats.loc[name, (m, 'std')]:.4f}"
                       for m in TABLE_METRICS)
    print(f"Pooled features & {LABELS[name]} & ... & {cells} \\\\")

## Per-label results

In [ ]:
per_label = pd.concat([pd.read_csv(RUN_RESULTS_DIR / f"{name}_fold_per_label.csv") for name in MODELS])

for metric in ["roc_auc", "f1", "precision", "recall"]:
    table = per_label.pivot_table(index="label", columns="model_name", values=metric)[MODELS]
    print(f"\n{metric}\n{table.loc[LABEL_COLUMNS].round(4).to_markdown()}")

## Paired comparison with the sequence models

Fold-by-fold difference in macro ROC-AUC against DPCNN and Bi-GRU (same folds).

In [ ]:
SEQUENCE_MODELS = ["dpcnn", "bigru_shared_max_attention"]

sequence = pd.concat([pd.read_csv(RESULTS_DIR / f"{name}_fold_metrics.csv") for name in SEQUENCE_MODELS
                      if (RESULTS_DIR / f"{name}_fold_metrics.csv").exists()])
scores = pd.concat([folds, sequence]).pivot_table(index="fold", columns="model_name", values="macro_roc_auc")

for pooled in MODELS:
    for other in [m for m in SEQUENCE_MODELS if m in scores]:
        diff = scores[other] - scores[pooled]
        print(f"{other} - {pooled}: {diff.mean():+.4f} ± {diff.std():.4f}, better on {(diff > 0).sum()}/5 folds")

diff = scores["fasttext_lightgbm"] - scores["fasttext_logistic_regression"]
print(f"LightGBM - LR: {diff.mean():+.4f} ± {diff.std():.4f}, better on {(diff > 0).sum()}/5 folds")